<a href="https://colab.research.google.com/github/OluwatifeAkinsola/RAG-Chatbot/blob/main/Building_SLM_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Detailed explanation:
1. torch: the main pytorch library for tensors and neural networks
2. torch.nn: contains building blocks like layers, loss functions
3. torch.nn.functional: gives functions like softmax, cross_entropy
4. numpy: for array operations(used in data handling)
5. math: for mathematical constant like sqrt
6.tqdm: shows a progress bar during training

In [ ]:
#Setup and imports libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from tqdm import tqdm


Detailed explanation 2 on Model Architecture:

1. vocab_size: each token is a number from 0 to 50256. The model outputs probabilities for each.

2. block_size: how many tokens the model looks at at once. 128 is small but OK for learning.

3. embed_dim: each token becomes a vector of 256 numbers (like a fingerprint).

4. num_heads: in multi‑head attention, we use 4 parallel attention mechanisms.

5. n_layers: we stack 4 transformer blocks to make the model deeper.

6. dropout: randomly turns off 10% of neurons during training to prevent overfitting.

In [ ]:
#Hyperparameters(model config)
#model architecture

vocab_size = 50257 #GPT-2 tokenizer vocabulary size
block_size = 128   #max context length for predictions(context size/maximum sequence length)
embed_dim =256     #size of token embeddings
num_heads = 4      # number of attention Heads
num_layers = 4     #number of transformer layers
dropout = 0.1      #dropout for regularization

#Training
batch_size =32      #number of sequences per training batch
learning_rate = 3e-4 #learning rate for optimizer
epochs= 5             #number of full passes over the dataset

In [ ]:
#step2: Data preparation(Tokenization). we willuse GPT-2 tokenizer
!pip install tiktoken datasets

In [ ]:
import tiktoken
from datasets import load_dataset

# Load tokenizer
enc = tiktoken.get_encoding("gpt2")

# Load a small subset of TinyStories for speed
ds = load_dataset("roneneldan/TinyStories", split="train")
ds = ds.select(range(5000))  # use only 5000 stories for quick training

# Tokenize all stories into one long list of token IDs
all_ids = []
for example in ds:
    ids = enc.encode_ordinary(example["text"])
    all_ids.extend(ids)
    all_ids.append(enc.eot_token)  # add end‑of‑text token after each story

# Convert to numpy array for easy slicing
all_ids = np.array(all_ids, dtype=np.uint16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Detailed Explanation:

1. enc.encode_ordinary(): turns a string into a list of integers (token IDs).

2. enc.eot_token: special token (50256) that marks the end of text.

3. We concatenate all stories into one long array all_ids. This is standard for language model training.

4. np.uint16 uses 2 bytes per token (fits because max token is 50256).



In [ ]:
# Now split into input/target sequence
def get_batches(data, block_size, batch_size):
    # 'data' is the 1D numpy array of token IDs (e.g., all_ids)

    # Calculate the number of possible starting positions for an (input, target) pair
    # For an input sequence of length `block_size` and a target sequence of length `block_size` (shifted by 1),
    # we need `block_size + 1` tokens. So, the last valid starting index is `len(data) - (block_size + 1)`.
    total_possible_start_indices = len(data) - (block_size + 1) + 1

    # The number of full batches we can generate
    num_batches = total_possible_start_indices // batch_size

    # Iterate through each full batch
    for i in range(num_batches):
        x_batch = np.zeros((batch_size, block_size), dtype=data.dtype)
        y_batch = np.zeros((batch_size, block_size), dtype=data.dtype)

        for k in range(batch_size):
            # Calculate the starting index for the k-th sequence within the current batch
            # This ensures contiguous batches of sequences from the 1D data
            sequence_start_idx = (i * batch_size) + k

            # Extract the input sequence (from start_idx to start_idx + block_size - 1)
            x_batch[k] = data[sequence_start_idx : sequence_start_idx + block_size]

            # Extract the target sequence (from start_idx + 1 to start_idx + block_size)
            y_batch[k] = data[sequence_start_idx + 1 : sequence_start_idx + block_size + 1]

        # Yield the batch as PyTorch tensors
        yield torch.tensor(x_batch), torch.tensor(y_batch)

# The problematic train_data creation and reshaping lines have been removed
# as get_batches now directly handles the 1D all_ids array and generates batches.

Detailed Explanation
Explanation:

Language model training: given tokens [t1, t2, ..., t_block], predict [t2, t3, ..., t_{block+1}].

We slide a window of size block_size+1 over the data.

get_batches yields batches of input x and targets y for training.

In [ ]:
#Step 3 Positional Embedding since the transformer has no built-in sense of order, we add position information
class positionalEmbedding(nn.Module):
    def __init__(self, embed_dim, block_size):
         super().__init__()
         #Learnable embedding table for positions 0 to block_size-1
         self.pos_emb = nn.Embedding(block_size, embed_dim)
    def forward (self, x):
       #x shape: (batch, seq_len, emed_dim) - we ignore content, just need seq_len
        seq_len = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  #(1, seq_len)
        return self.pos_emb(positions) # (1, seq_len, embed_dim)

Detailed Explanation:

nn.Embedding(block_size, embed_dim): a table of size block_size × embed_dim. Row p gives the embedding for position p.

torch.arange(seq_len): creates tensor [0,1,2,...,seq_len-1].

.unsqueeze(0): adds a batch dimension (1, seq_len).

We'll add this to the token embeddings later.

In [ ]:
#Step 4: Causal Self‑Attention
#This is the heart of the transformer. It lets each token look at previous tokens (not future ones).
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5  # 1/sqrt(head_dim)

        # Single linear layer that produces Q, K, V for all heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # batch, time, channels (embed_dim)

        # Compute Q, K, V and split into heads
        qkv = self.qkv(x)  # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)  # each (B, T, C)

        # Reshape to (B, num_heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention scores: (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * self.scale

        # Causal mask: lower triangular matrix, -inf for future positions
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float('-inf'))

        # Softmax and dropout
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        # Apply attention to values
        y = att @ v  # (B, nh, T, head_dim)

        # Combine heads back to (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.proj(y)
        return y

Explanation of key lines:

qkv = self.qkv(x): one matrix multiplication produces all of Q, K, V at once (efficient).

.chunk(3, dim=-1): splits into three tensors along last dimension.

.view().transpose(): rearranges to separate heads.

att = (q @ k.transpose(-2,-1)) * self.scale: dot product and scaling.

torch.tril: creates a lower‑triangular matrix of ones. Positions above diagonal are zero → future tokens are masked.

att.masked_fill(mask==0, -inf): future positions become -inf, so softmax gives zero probability.

y = self.proj(y): final linear projection after concatenating heads.



In [ ]:
#Step 5: Feed‑Forward Network (MLP)
#After attention, each token passes through a small neural network.
class MLP(nn.Module):
    def __init__(self, embed_dim, expansion_factor=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * expansion_factor
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

Brief Explanation:

fc1 expands dimension from embed_dim to 4*embed_dim.

nn.GELU is a smooth activation function (like ReLU but smoother).

fc2 projects back to embed_dim.

Dropout prevents overfitting.

In [ ]:
#Step 6: Transformer Block
#Combine attention + MLP with residual connections and layer normalization.
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, expansion_factor=4, dropout=dropout)

    def forward(self, x):
        # Pre‑norm architecture (better for deep networks)
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

Bief Explanation:

LayerNorm normalizes activations to zero mean and unit variance – helps training.

Residual connection x + ... allows gradients to flow directly through the network.

Pre‑norm (apply layer norm before attention/MLP) is standard in modern transformers.

In [ ]:
#Step 7: Full SLM Model
#Stack token embeddings, positional embeddings, transformer blocks, and output layer.
class SmallLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, n_layers, block_size, dropout):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = positionalEmbedding(embed_dim, block_size) # Corrected to positionalEmbedding
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Tie weights: output projection shares with token embeddings (optional)
        self.token_emb.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.shape
        # Token embeddings
        tok_emb = self.token_emb(idx)  # (B, T, embed_dim)
        # Positional embeddings
        pos_emb = self.pos_emb(tok_emb)  # (1, T, embed_dim)
        x = tok_emb + pos_emb

        # Apply transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final layer norm
        x = self.ln_f(x)
        # Project to vocabulary
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits

Brief Explanation:

token_emb: maps token ID (0‑50256) to a vector of size embed_dim.

pos_emb: adds position information to each token.

blocks: list of transformer blocks.

ln_f: final layer norm for stability.

lm_head: linear layer that outputs logits for each token in the vocabulary.

Weight tying: token_emb.weight = self.lm_head.weight reduces parameters and often improves performance.

In [ ]:
#Step 8: Initialize Model, Optimizer, and Loss
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = SmallLanguageModel(vocab_size, embed_dim, num_heads, num_layers, block_size, dropout)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

Using device: cuda


Brief Explanation:

device: automatically uses GPU if available, otherwise CPU.

model.to(device): moves all model parameters to GPU/CPU.

AdamW: optimizer with weight decay (improves generalization).

CrossEntropyLoss: combines log_softmax and negative log likelihood – standard for classification.

In [ ]:
#Step 9: Training Loop
model.train()
for epoch in range(epochs):
    total_loss = 0
    num_batches = 0
    # Create data loader iterator
    data_iter = get_batches(all_ids, block_size, batch_size)
    pbar = tqdm(data_iter, desc=f"Epoch {epoch+1}")
    for x, y in pbar:
        x, y = x.to(device).long(), y.to(device).long() # Cast to torch.long
        # Forward pass
        logits = model(x)               # (batch, seq_len, vocab)
        # Reshape logits and targets for loss calculation
        loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': loss.item()})
    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

Epoch 1: 32279it [1:32:52,  5.79it/s, loss=4.01]


Epoch 1 average loss: 4.3120


Epoch 2: 19814it [57:07,  5.80it/s, loss=3.26]

Brief Explanation:

model.train(): sets model to training mode (enables dropout).

get_batches: yields batches of input and target tensors.

logits.view(-1, vocab_size): flattens batch and time dimensions into one axis.

loss.backward(): computes gradients.

clip_grad_norm_: prevents gradients from becoming too large (stabilizes training).

optimizer.step(): updates model weights.

tqdm shows progress bar with current loss.



In [ ]:
#Step 10: Text Generation (Sampling)
#After training, we can generate new text.
@torch.no_grad()
def generate(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    model.eval()
    # Tokenize the prompt
    input_ids = enc.encode_ordinary(prompt)
    input_tensor = torch.tensor(input_ids).unsqueeze(0).to(device)  # (1, T)

    for _ in range(max_new_tokens):
        # Crop to block_size if needed
        if input_tensor.size(1) > block_size:
            input_tensor = input_tensor[:, -block_size:]

        # Get logits for the last token
        logits = model(input_tensor)  # (1, T, vocab)
        logits = logits[0, -1, :] / temperature  # scale by temperature

        # Optional top‑k sampling
        if top_k is not None:
            indices = torch.topk(logits, top_k).indices
            logits = logits.scatter(0, indices, logits.gather(0, indices))
            logits = torch.where(logits < logits.min(), float('-inf'), logits)

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]]).to(device)], dim=1)

    output_ids = input_tensor[0].tolist()
    output_text = enc.decode(output_ids)
    return output_text

Brief Explanation:

model.eval(): disables dropout for inference.

temperature: higher = more random; lower = more deterministic.

top_k: keep only top k highest probability tokens, discard the rest.

torch.multinomial: samples a token according to the probability distribution.

We append the new token to the input and repeat.

In [ ]:
#Step 11: Try Generation
prompt = "Once upon a time"
generated = generate(model, prompt, max_new_tokens=100, temperature=0.7, top_k=40)
print(generated)
torch.save(model.state_dict(), 'slm_model.pt')